In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip -q install faiss-cpu sentence-transformers transformers

import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.4 MB/s eta 0:00:00


In [3]:
train_df="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
train = pd.read_csv(train_df)

print("Creating Knowledge Base...")

kb = []

for _, row in train.iterrows():
    correct_letter = row["answer"]
    kb.append(str(row[correct_letter]))

print("Loading embedding model...")

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

kb_embeddings = embed_model.encode(
    kb,
    convert_to_numpy=True,
    show_progress_bar=True
)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge Base Created Successfully!")

Creating Knowledge Base...
Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Knowledge Base Created Successfully!


In [4]:
print("Loading Zero-Shot Classifier...")

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

print("Loading Cross Encoder...")

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

Loading Zero-Shot Classifier...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
# q1
row = train.iloc[150]

prompt = str(row["prompt"])

labels = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

true_answer = str(row[row["answer"]])

result = zs(prompt, candidate_labels=labels)

score = result["scores"][result["labels"].index(true_answer)]

print(round(score,3))

0.384


In [6]:
# q2
query_embedding = embed_model.encode(
    [prompt],
    convert_to_numpy=True
)

D, I = index.search(query_embedding, 10)

retrieved_indices = I[0]

true_rank = list(retrieved_indices).index(150) + 1

print(true_rank)

10


In [7]:
# q3
docs = [kb[i] for i in retrieved_indices]

pairs = [[prompt, doc] for doc in docs]

scores = cross_encoder.predict(pairs)

order = np.argsort(scores)[::-1]

reranked = retrieved_indices[order]

true_rank = list(reranked).index(150) + 1

print(true_rank)

1


In [8]:
# q4
row = train.iloc[42]

prompt = str(row["prompt"])

query_embedding = embed_model.encode(
    [prompt],
    convert_to_numpy=True
)

_, I = index.search(query_embedding, 5)

docs = [kb[i] for i in I[0]]

context = " ".join(docs)

rag = f"Context: {context} Question: {prompt}"

tokens = tokenizer(
    rag,
    add_special_tokens=True,
    truncation=False
)

print(len(tokens["input_ids"]))

216


In [9]:
# q5
row = train.iloc[150]

prompt = str(row["prompt"])

true_doc = kb[150]

rag = f"Context: {true_doc} Question: {prompt}"

labels = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

true_answer = str(row[row["answer"]])

result = zs(rag, candidate_labels=labels)

score = result["scores"][result["labels"].index(true_answer)]

print(round(score,3))

0.989


In [10]:
# q6
wrong_doc = kb[999]

rag = f"Context: {wrong_doc} Question: {prompt}"

result = zs(rag, candidate_labels=labels)

score = result["scores"][result["labels"].index(true_answer)]

print(round(score,3))

0.529


In [11]:
# q7
hits = 0

for idx in range(100):

    row = train.iloc[idx]

    prompt = str(row["prompt"])

    true_doc = str(row[row["answer"]])

    emb = embed_model.encode(
        [prompt],
        convert_to_numpy=True
    )

    _, I = index.search(emb, 5)

    retrieved = [kb[i] for i in I[0]]

    if any(true_doc == doc for doc in retrieved):
        hits += 1

hit_rate = hits / 100 * 100

print(round(hit_rate,1))

73.0


In [12]:
# q8
def map3(actual, predicted):

    score = 0

    for k, pred in enumerate(predicted[:3], start=1):
        if pred == actual:
            score = 1 / k
            break

    return score


scores = []

for idx in range(20):

    row = train.iloc[idx]

    prompt = str(row["prompt"])

    emb = embed_model.encode(
        [prompt],
        convert_to_numpy=True
    )

    _, I = index.search(emb, 5)

    retrieved = I[0]

    docs = [kb[i] for i in retrieved]

    pairs = [[prompt, d] for d in docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce_scores)]

    rag = f"Context: {best_doc} Question: {prompt}"

    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    result = zs(rag, candidate_labels=labels)

    ranked = []

    for label in result["labels"]:
        if label == str(row["A"]):
            ranked.append("A")
        elif label == str(row["B"]):
            ranked.append("B")
        elif label == str(row["C"]):
            ranked.append("C")
        elif label == str(row["D"]):
            ranked.append("D")
        elif label == str(row["E"]):
            ranked.append("E")

    scores.append(
        map3(
            row["answer"],
            ranked
        )
    )

print(round(np.mean(scores),3))

0.975
